MEMO Debugging

In [1]:
import torch
from MeMoHF.modelling_memo_tokenizer import MeMoTokenizer
from MeMoHF.modelling_memo_configuration import MeMoConfig
from MeMoHF.modelling_memo import MeMoForCausalLM
from MeMoHF.evaluating_memo import Evaluation
from MeMoHF.utils import seed_everything

seed_everything(42)

# d, h, l = 1024, 4, 4
# chunk_length = 256

d, h, l = 1024, 4, 2
chunk_length = 16
#d,h,l = 2048, 42, 2
#chunk_length =  (h ** l)

# Initializing a standard Tokenizer
max_length = chunk_length 
tokenizer = MeMoTokenizer.from_pretrained("EleutherAI/gpt-neox-20b", 
                                          padding_side='left', truncation_side='left', 
                                          model_max_length=max_length, head_number=h)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.pad_token_id

# Intializing Memo Configuration
config = MeMoConfig(vocab_size=len(tokenizer), #tokenizer.vocab_size, 
    hidden_size=d, 
    num_hidden_layers=l,
    num_attention_heads=h,
    chunk_length=chunk_length,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id,
    compositionOp='ProdWithShuffling',
#    compositionOp='sum',
    padding_seq_idx=tokenizer.pad_token_id,
    padding_vector_component_values=1/(d**(1/2))  #new!

)

# Initializing the Memo Model from the configuration

model = MeMoForCausalLM(config) 
model.training = True
model.train()

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} is available.")
    model.to('cuda')


C:\Program Files (Arm)\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'GPTNeoXTokenizer'. 
The class this function is called from is 'MeMoTokenizer'.


Setting pad token and pad token id = <|endoftext|>, 0
Padding input tokens in MeMo architecture:  0.03125
MeMoEmbedding 0.03125
MeMoEmbedding 0
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0.03125
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0.03125
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0
MeMo embedding initilialization:  MeMoEmbedding
Padding vector components :  0


In [2]:
texts = [
            ' a b c d e f g h i l m n o p q r r',
            ' a b c b e f g h i l m n o p q r o', # CAMBIO DI UNA LETTERA PRIMO BLOCCO
            ' a b b d e f g h i l m n o p q r x', # CAMBIO DI UNA LETTERA PRIMO BLOCCO
            ' c b a d e f g h i l m n o p q r w', # INVERSIONE LETTERA NEL PRIMO BLOCCO
            ' c a b d e f g h i l m n o p q r r', # INVERSIONE LETTERA NEL PRIMO BLOCCO
            ' a a a d e f g h i l m n o p q r j', # ripetizione nel primo blocco
            ' a b c d e f w h i l m n o p q r l'  # CAMBIO DI UNA LETTERA
          ]

memo_inputs = []
for text in texts:
    memo_input = tokenizer.get_text_batch_encoding_for_loss([text]*1)
    print(text)
    print(memo_input['input_ids'], len(memo_input['input_ids'][0]))
    print(memo_input['labels'], len(memo_input['input_ids'][0]))
    memo_inputs.append(memo_input)




 a b c d e f g h i l m n o p q r r
tensor([[ 247,  270,  260,  277,  299,  269,  305,  288,  891,  298,  278,  295,
          258,  268, 2805,  391]]) 16
tensor([[ 270,  260,  277,  299,  269,  305,  288,  891,  298,  278,  295,  258,
          268, 2805,  391,  391]]) 16
 a b c b e f g h i l m n o p q r o
tensor([[ 247,  270,  260,  270,  299,  269,  305,  288,  891,  298,  278,  295,
          258,  268, 2805,  391]]) 16
tensor([[ 270,  260,  270,  299,  269,  305,  288,  891,  298,  278,  295,  258,
          268, 2805,  391,  258]]) 16
 a b b d e f g h i l m n o p q r x
tensor([[ 247,  270,  270,  277,  299,  269,  305,  288,  891,  298,  278,  295,
          258,  268, 2805,  391]]) 16
tensor([[ 270,  270,  277,  299,  269,  305,  288,  891,  298,  278,  295,  258,
          268, 2805,  391, 1269]]) 16
 c b a d e f g h i l m n o p q r w
tensor([[ 260,  270,  247,  277,  299,  269,  305,  288,  891,  298,  278,  295,
          258,  268, 2805,  391]]) 16
tensor([[ 270,  247,  277, 

In [3]:
# evaluation method
from MeMoHF.evaluating_memo import EvaluationUpdateNew
evaluation = EvaluationUpdateNew()

def perform_evaluation(model=None, tokenizer=None, text=None, starting_point=1):
    model.eval()
    batch_inputs = tokenizer.get_text_batch_encoding_for_loss(text=text) #for_loss
    #print(batch_inputs)
    with torch.no_grad():
        actual_output = model.forward_with_loss_simple( 
            #model.forward_with_loss_parallelized_efficient( #model.forward_with_loss_parallelized_efficient( #model.forward_with_loss_parallelized(
            batch_inputs=batch_inputs,
            compute_accuracy=True,
            return_dict=True,
            starting_point=3
        )
        print(actual_output)

        outs, pretokenized_score, _ = actual_output

    loss = outs['loss']
    del outs
    model.train()
    torch.cuda.empty_cache()
    return dict(
        out_loss=loss,
        token_accuracy=pretokenized_score
    )

In [4]:
for memo_input in memo_inputs:
    model.memorize_text(memo_input)

for memo_input in memo_inputs:
    o = model.retrieve(memo_input['input_ids'])
    #print()
    print(f"SEQ  {tokenizer.batch_decode(memo_input['input_ids'])}")
    print(f"ExOUT  {tokenizer.batch_decode(memo_input['labels'])}")
    print(f"   OUT {tokenizer.batch_decode(o.logits.max(dim=-1).indices)}")


#print("EVALUATION ")
#for text in texts:
#    print(text)

#    o.logits.max(dim=-1)

#    results_1 = perform_evaluation(
#        model=model,
#        tokenizer=tokenizer,
#        text=[text]*1
#    )
#    print(f'Accuracy: {results_1["token_accuracy"]["accuracy"]}')


Decoding input_sequence at layer 0
(tensor([[[   0,    0,    0,  247],
         [   0,    0,  247,  270],
         [   0,  247,  270,  260],
         [ 247,  270,  260,  277],
         [ 270,  260,  277,  299],
         [ 260,  277,  299,  269],
         [ 277,  299,  269,  305],
         [ 299,  269,  305,  288],
         [ 269,  305,  288,  891],
         [ 305,  288,  891,  298],
         [ 288,  891,  298,  278],
         [ 891,  298,  278,  295],
         [ 298,  278,  295,  258],
         [ 278,  295,  258,  268],
         [ 295,  258,  268, 2805],
         [ 258,  268, 2805,  391]]]), tensor([[[1.0000, 1.0000, 1.0000, 1.0064],
         [1.0000, 1.0000, 1.0064, 1.0493],
         [1.0000, 1.0064, 1.0493, 1.0178],
         [1.0064, 1.0493, 1.0178, 1.0198],
         [1.0493, 1.0178, 1.0198, 0.9019],
         [1.0178, 1.0198, 0.9019, 0.9689],
         [1.0198, 0.9019, 0.9689, 0.9944],
         [0.9019, 0.9689, 0.9944, 0.9854],
         [0.9689, 0.9944, 0.9854, 1.0004],
         [0.99

In [5]:
exit()